# 05.1 Evaluation: Frozen Phase Sync Checkpoint

Evaluate the phase-synchronization fine-tuned diffusion policies produced by `06_frozen_phase_estimator.ipynb`. This notebook keeps `05_evaluation.ipynb` unchanged and adds all trained phase-sync lambda ablations as extra trajectory models.

In [ ]:
# Google Drive mount and project-root setup for Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

PROJECT_DIR = Path('/content/drive/MyDrive/phase_conditioned_diffusion_policy')

try:
    from google.colab import drive
except ImportError:
    print(f'Not running in Google Colab; keeping current working directory: {Path.cwd()}')
else:
    drive.mount('/content/drive')
    if not PROJECT_DIR.exists():
        raise FileNotFoundError(
            f'Expected project directory not found: {PROJECT_DIR}\n'
            'Update PROJECT_DIR to the Google Drive folder that contains this repository.'
        )
    os.chdir(PROJECT_DIR)
    print(f'Current working directory: {Path.cwd()}')

# Colab dependency setup
if importlib.util.find_spec('google.colab') is None:
    print('Not running in Google Colab; skipping dependency installation and using the current environment.')
else:
    project_root = Path.cwd()
    requirements_path = project_root / 'requirements.txt'
    install_command = [sys.executable, '-m', 'pip', 'install']
    if requirements_path.exists():
        install_command.extend(['-r', str(requirements_path), '-e', str(project_root)])
    else:
        install_command.extend(['-e', str(project_root)])
    subprocess.check_call(install_command)

    import gymnasium as gym
    import mujoco
    import minari
    import torch

    gym.make('Ant-v5').close()
    print(f'gymnasium={gym.__version__}')
    print(f'mujoco={mujoco.__version__}')
    print(f'minari={minari.__version__}')
    print(f'torch={torch.__version__}')
    print('Ant-v5 environment smoke check passed.')

## 1. Imports and Artifact Paths

In [ ]:
from dataclasses import replace
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch

import pcdp.evaluation as eval_mod
from pcdp.configs import get_experiment_config, set_global_seed
from pcdp.dataset import load_project_data
from pcdp.evaluation import (
    EvaluationState,
    LoadedEvalModel,
    ModelEvalSpec,
    build_eval_results_payload,
    build_frequency_sweep_protocol,
    load_evaluation_state,
    metric_array,
    print_frequency_sweep_summary,
    print_table1_summary,
    run_frequency_sweep_evaluation,
    run_in_distribution_evaluation,
    save_eval_results_npz,
    write_frequency_tracking_table_markdown,
    write_table1_summary_markdown,
)
from pcdp.frozen_phase_estimator import load_phase_sync_checkpoint
from pcdp.paths import ARTIFACT_ROOT, CHECKPOINTS_DIR, DATA_DIR, FIGURES_DIR, RESULTS_DIR, ensure_artifact_dirs
from pcdp.training import load_checkpoint

ensure_artifact_dirs()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}, device={device}')
print(f'artifact root: {ARTIFACT_ROOT}')
print(f'checkpoints: {CHECKPOINTS_DIR}')

## 2. Load Data and Base Evaluation State

In [ ]:
data_dir = DATA_DIR
print(f'data_dir={data_dir}')

data = load_project_data(data_dir)
set_global_seed(data['seed'], deterministic=True)
state = load_evaluation_state(data, device=device, checkpoints_dir=CHECKPOINTS_DIR)

## 3. Add Phase-Sync Checkpoints

In [ ]:
SYNC_SPECS = [
    ('trajectory_sync_l001', 'Trajectory + Sync lambda=0.01', 'phase_trajectory_sync_lambda0.01.pt'),
    ('trajectory_sync_l002', 'Trajectory + Sync lambda=0.02', 'phase_trajectory_sync_lambda0.02.pt'),
    ('trajectory_sync_l005', 'Trajectory + Sync lambda=0.05', 'phase_trajectory_sync_lambda0.05.pt'),
    ('trajectory_sync_l010', 'Trajectory + Sync lambda=0.1',  'phase_trajectory_sync_lambda0.1.pt'),
]

missing = [name for _key, _label, name in SYNC_SPECS if not (CHECKPOINTS_DIR / name).exists()]
assert not missing, (
    'Missing sync checkpoints:\n  - ' + '\n  - '.join(missing) +
    '\nRun notebooks/06_frozen_phase_estimator.ipynb for each lambda first.'
)

sync_cfg = get_experiment_config('phase_trajectory')
loaded_models = dict(state.loaded_models)
model_specs = dict(state.model_specs)

for sync_key, sync_label, sync_ckpt_name in SYNC_SPECS:
    sync_ckpt_path = CHECKPOINTS_DIR / sync_ckpt_name
    sync_model = sync_cfg.build_model(data, device=device)
    sync_ema = sync_cfg.build_ema(sync_model)
    load_phase_sync_checkpoint(sync_ckpt_path, sync_model, sync_ema, device=device, use_best_ema=True)

    loaded_models[sync_key] = LoadedEvalModel(
        config_name=sync_key,
        label=sync_label,
        model=sync_model,
        ema=sync_ema,
        cond_fn=sync_cfg.resolve_sample_cond_fn(),
    )
    model_specs[sync_key] = ModelEvalSpec(
        loaded_name=sync_key,
        label=sync_label,
        model=sync_model,
        ema=sync_ema,
        cond_fn=sync_cfg.resolve_sample_cond_fn(),
        uses_phase_trajectory=True,
    )

state = EvaluationState(
    configs=state.configs,
    loaded_models=loaded_models,
    model_specs=model_specs,
    noise_scheduler_config=state.noise_scheduler_config,
    num_inference_steps=state.num_inference_steps,
)

# Existing evaluation/table helpers iterate over module-level model lists.
sync_keys = tuple(key for key, _label, _ckpt_name in SYNC_SPECS)
eval_mod.MODEL_KEYS = ('vanilla', 'periodic', 'trajectory', *sync_keys)
eval_mod.PHASE_MODEL_KEYS = ('periodic', 'trajectory', *sync_keys)
for sync_key, sync_label, _sync_ckpt_name in SYNC_SPECS:
    eval_mod.SUMMARY_MODEL_LABELS[sync_key] = sync_label
    eval_mod.PHASE_CONDITION_LABELS[sync_key] = 'full phase trajectory + sync loss'

print('Models for Table 1:', eval_mod.MODEL_KEYS)
print('Models for Table 2:', eval_mod.PHASE_MODEL_KEYS)

## 4. Evaluation Protocol

In [ ]:
DT = 0.05
MAX_STEPS = 1000
N_SEEDS_INDIST = 20
N_SEEDS_SWEEP = 10

freq_protocol = build_frequency_sweep_protocol(data, n_in_dist=3, ood_iqr_scale=1.5)
print(f'In-dist freqs: {freq_protocol.in_freqs.round(3).tolist()}')
print(f'OOD freqs:     {freq_protocol.ood_freqs.round(3).tolist()}')
print(f'Sweep freqs:   {freq_protocol.sweep_freqs.round(3).tolist()}')
print(f'Zones:         {freq_protocol.zone_labels.tolist()}')

## 5. Table 1: In-Distribution Rollout Quality

In [ ]:
env = gym.make('Ant-v5')
table1_results = run_in_distribution_evaluation(
    state,
    env=env,
    data=data,
    device=device,
    n_seeds=N_SEEDS_INDIST,
    max_steps=MAX_STEPS,
    dt=DT,
)
print_table1_summary(state, table1_results, freq_hz=float(data['freq_window_mean']))
write_table1_summary_markdown(
    state,
    table1_results,
    RESULTS_DIR / 'table1_indist_quality_with_phase_sync.md',
    freq_hz=float(data['freq_window_mean']),
    interval='ci95',
)

## 6. Table 2: Frequency Command Tracking

In [ ]:
freq_results = run_frequency_sweep_evaluation(
    state,
    freq_protocol,
    env=env,
    data=data,
    device=device,
    model_keys=eval_mod.PHASE_MODEL_KEYS,
    n_seeds=N_SEEDS_SWEEP,
    max_steps=MAX_STEPS,
    dt=DT,
)
print_frequency_sweep_summary(data, freq_protocol, freq_results)
write_frequency_tracking_table_markdown(
    freq_protocol,
    freq_results,
    RESULTS_DIR / 'table2_frequency_tracking_with_phase_sync.md',
    interval='ci95',
)

## 7. Phase-Sync Comparison Figures

In [ ]:
def _mean_std(values):
    arr = np.asarray(values, dtype=np.float32)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.nan, np.nan
    return float(arr.mean()), float(arr.std())

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
model_keys = list(eval_mod.MODEL_KEYS)
phase_keys = list(eval_mod.PHASE_MODEL_KEYS)
colors = {
    'vanilla': 'tab:blue',
    'periodic': 'tab:green',
    'trajectory': 'tab:red',
    'trajectory_sync_l001': 'tab:purple',
    'trajectory_sync_l002': 'tab:brown',
    'trajectory_sync_l005': 'tab:pink',
    'trajectory_sync_l010': 'tab:cyan',
}

# Figure A: in-distribution reward per step for baselines and sync ablations.
labels = [state.model_specs[k].label for k in model_keys]
means, stds = [], []
for key in model_keys:
    m, s = _mean_std(metric_array(table1_results[key], 'reward_per_step'))
    means.append(m)
    stds.append(s)

fig, ax = plt.subplots(figsize=(max(8.2, 1.35 * len(model_keys)), 4.8))
x = np.arange(len(model_keys))
ax.bar(x, means, yerr=stds, capsize=4, color=[colors[k] for k in model_keys], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha='right')
ax.set_ylabel('Reward / step')
ax.set_title('In-distribution reward per step with phase-sync ablations')
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
out = FIGURES_DIR / 'eval_05_1_reward_per_step_with_phase_sync.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
print(f'saved {out}')

# Figure B: commanded vs measured frequency for phase-conditioned models.
freqs = sorted(float(f) for f in freq_results['trajectory'].keys())
fig, ax = plt.subplots(figsize=(7.4, 5.8))
for key in phase_keys:
    y, yerr = [], []
    for freq in freqs:
        m, s = _mean_std(metric_array(freq_results[key][freq], 'measured_freq_hz'))
        y.append(m)
        yerr.append(s)
    ax.errorbar(freqs, y, yerr=yerr, marker='o', capsize=4, linewidth=2, label=state.model_specs[key].label, color=colors[key])
lo = min(min(freqs), float(data['freq_window_min']))
hi = max(max(freqs), float(data['freq_window_max']))
pad = max((hi - lo) * 0.08, 0.02)
diag = np.linspace(lo - pad, hi + pad, 100)
ax.plot(diag, diag, '--', color='black', alpha=0.45, label='perfect tracking')
ax.axvspan(float(data['freq_window_min']), float(data['freq_window_max']), alpha=0.12, color='green', label='in-dist range')
ax.set_xlim(lo - pad, hi + pad)
ax.set_ylim(lo - pad, hi + pad)
ax.set_xlabel('Commanded frequency (Hz)')
ax.set_ylabel('Measured gait frequency (Hz)')
ax.set_title('Frequency command tracking with phase-sync ablations')
ax.legend(fontsize=8, ncol=1)
ax.grid(True, alpha=0.3)
fig.tight_layout()
out = FIGURES_DIR / 'eval_05_1_frequency_tracking_with_phase_sync.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
print(f'saved {out}')

## 8. Save Raw Results

In [ ]:
eval_payload = build_eval_results_payload(
    data,
    table1_results,
    freq_protocol,
    freq_results,
    n_seeds_indist=N_SEEDS_INDIST,
    n_seeds_sweep=N_SEEDS_SWEEP,
)
eval_payload['sync_checkpoint_names'] = np.asarray([ckpt_name for _key, _label, ckpt_name in SYNC_SPECS])
save_eval_results_npz(eval_payload, RESULTS_DIR / 'eval_results_with_phase_sync.npz')